# PoA Upper Bound via Gurobi (for general d≥3)

This notebook loads the code package in `poa_bounds/` and computes a smoothness-based PoA upper bound.

In [ ]:
import sys, os
import numpy as np
from poa_bounds import poa_upper_bound

def run_adaptive_search():
    params = {
        "n": 5, "p": 0.55, "d": 5, 
        "rule": "sv", "ps_model": "explicit",
        "verbose": False, "max_workers": 8, "time_limit_per": 10
    }


    print("Step 1: Global scan (20 points)...")
    mu_grid_global = np.linspace(0.1, 0.99, 10).tolist()
    res_global = poa_upper_bound(mu_grid=mu_grid_global, **params)
    
    best_mu = res_global["best"]["mu"]
    print(f"Current best mu: {best_mu:.4f}, PoA: {res_global['best']['PoA']:.4f}")


    print("\nStep 2: Fine-grained local scan (20 points)...")
    mu_min = max(0.01, best_mu - 0.05)
    mu_max = min(0.99, best_mu + 0.05)
    mu_grid_fine = np.linspace(mu_min, mu_max, 20).tolist()
    
    res_fine = poa_upper_bound(mu_grid=mu_grid_fine, **params)
    
    print("\n" + "="*30)
    print(f"Final PoA Upper Bound: {res_fine['best']['PoA']:.4f}")
    print(f"Optimal mu*: {res_fine['best']['mu']:.6f}")
    print(f"Lambda(mu*): {res_fine['best']['Lambda']:.4f}")
    print("="*30)

if __name__ == "__main__":
    run_adaptive_search()

In [ ]:
import time
import numpy as np
import pandas as pd
from poa_bounds import poa_upper_bound

from datetime import datetime
from zoneinfo import ZoneInfo

run_id = datetime.now(ZoneInfo("Asia/Singapore")).strftime("%Y%m%d_%H%M%S")
fname_sv = f"results_sv_{run_id}.csv"
fname_ps = f"results_ps_{run_id}.csv"
fname_all = f"results_all_{run_id}.csv"


def run_full_experiments():
    d_list = [3, 4, 5]
    n_list = [3, 5, 10]
    p_list = np.linspace(0.05, 1.0, 20).tolist()
    rules = ["ps", "sv"]
    results = []

    try:
        for rule in rules:
            for d in d_list:
                for n in n_list:
                    for p in p_list:
                        params = {
                            "n": n,
                            "p": p,
                            "d": d,
                            "rule": rule,
                            "verbose": False,
                            "time_limit_per": 10,
                        }

                        start_time = time.time()

                        # Global scan: take 20 uniform points on (0, 1)
                        mu_grid_global = np.linspace(0.1, 0.99, 20).tolist()
                        res_global = poa_upper_bound(mu_grid=mu_grid_global, **params)
                        best_mu_global = res_global["best"]["mu"]

                        # Local fine scan: scan 20 points around coarse optimum
                        mu_min = max(0.05, best_mu_global - 0.05)
                        mu_max = min(0.99, best_mu_global + 0.05)
                        mu_grid_fine = np.linspace(mu_min, mu_max, 20).tolist()
                        res_fine = poa_upper_bound(mu_grid=mu_grid_fine, **params)

                        end_time = time.time()

                        arg = res_fine["best"].get("argmax") or {}
                        nD_star = arg.get("nD", np.nan)
                        nI_star = arg.get("nI", np.nan)

                        results.append({
                            "rule": rule,
                            "d": d,
                            "n": n,
                            "p": p,
                            "mu_star": res_fine["best"]["mu"],
                            "nD_star": nD_star,
                            "nI_star": nI_star,
                            "PoA": res_fine["best"]["PoA"],
                            "calc_time_sec": end_time - start_time,
                        })

                        print(
                            f"Done: rule={rule}, d={d}, n={n}, p={p:.2f} | "
                            f"PoA={res_fine['best']['PoA']:.4f}"
                        )

                        # Save progress in real time to prevent data loss
                        df_temp = pd.DataFrame(results)
                        df_temp[df_temp["rule"] == "ps"].to_csv(fname_ps, index=False)
                        df_temp[df_temp["rule"] == "sv"].to_csv(fname_sv, index=False)

    except KeyboardInterrupt:
        print("\nExperiment interrupted! Saving completed data...")

    finally:
        # Ensure final save upon completion or interruption
        if results:
            df_temp = pd.DataFrame(results)
            df_temp.to_csv(fname_all, index=False)

            if "ps" in rules:
                df_temp[df_temp["rule"] == "ps"].to_csv(fname_ps, index=False)
            if "sv" in rules:
                df_temp[df_temp["rule"] == "sv"].to_csv(fname_sv, index=False)

            print("All available data successfully saved.")
        else:
            print("No computations completed. No data saved.")


if __name__ == "__main__":
    run_full_experiments()